# Crop disease: preserve baseline, adapt to field images, evaluate honestly

This notebook continues the existing **EfficientNet-B0 / 38-class** project. It does not replace the reference checkpoint. It downloads PlantDoc and PlantWild v1, maps compatible labels into the existing class IDs, holds out deduplicated in-the-wild images for the final field-like test, fine-tunes from the baseline, and reports PlantVillage and field-domain results separately.

An optional EfficientNet-B0 decoder can train for **diseased-region segmentation** using PlantSeg masks; classification fine-tuning can run without it. PlantSeg masks label diseased parts, not the full leaf outline; this is lesion segmentation/visual localization.

**Before running:** The original hardware-check and sparse PlantVillage download cells are retained. They recreate the 38-class map and 70/15/15 split manifests exactly as the prior Colab workflow did. If needed, setup asks you to upload the original `best_model.pth` downloaded from the earlier run; it never trains a replacement baseline. PlantVillage is written to `/content/data/raw/plantvillage` and splits to `/content/data/splits`. Baseline is copied and never overwritten. PlantWild v1 is CC BY-NC-ND 4.0 (non-commercial); check licenses and image-level rights before external/commercial use. PlantDoc is web-scraped and its repository states CC BY 4.0. Sources: [PlantDoc](https://github.com/pratikkayal/PlantDoc-Dataset), [PlantWild](https://tqwei05.github.io/PlantWild/), [PlantSeg](https://github.com/tqwei05/PlantSeg).

In [ ]:
# ==========================================
# Step 1: Verify Hardware & GPU Acceleration
# ==========================================
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Target Device:", torch.cuda.get_device_name(0))
    print("Memory Allocated:", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise SystemError("GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")

from pathlib import Path
import os, json, random, shutil, hashlib, zipfile, subprocess, sys, re, math, time
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda')

# Keep the checkpoint path used by the original Colab workflow.
BASELINE_CKPT = Path('/content/checkpoints/best_model.pth')
if not BASELINE_CKPT.is_file():
    from google.colab import files
    print(f'Upload the original trained best_model.pth; saving to {BASELINE_CKPT}')
    uploaded = files.upload()
    if 'best_model.pth' not in uploaded:
        raise FileNotFoundError('Select the original best_model.pth downloaded from your previous run.')
    BASELINE_CKPT.parent.mkdir(parents=True, exist_ok=True)
    BASELINE_CKPT.write_bytes(uploaded['best_model.pth'])

DATA = Path('/content/data')
RAW = DATA / 'raw'
PV = RAW / 'plantvillage'
SPLITS_DIR = DATA / 'splits'
BASELINE_CLASSES = SPLITS_DIR / 'classes.json'
ORIGINAL_SPLITS = SPLITS_DIR
WORK = Path('/content/cropdisease_adaptation')
OUT = WORK / 'outputs'
for p in (RAW, SPLITS_DIR, OUT): p.mkdir(parents=True, exist_ok=True)
BASELINE_BACKUP = OUT / 'baseline_best_model.pth'
if not BASELINE_BACKUP.exists(): shutil.copy2(BASELINE_CKPT, BASELINE_BACKUP)
print('Original checkpoint preserved at:', BASELINE_BACKUP)


In [ ]:
# ==========================================
# Step 2: Fast Sparse Download of PlantVillage
# ==========================================
!mkdir -p data/raw
!rm -rf temp_pv
!echo "Cloning color images sparse checkout..."
!git clone --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git temp_pv
!cd temp_pv && git sparse-checkout set raw/color
!mv temp_pv/raw/color data/raw/plantvillage
!rm -rf temp_pv

import os
classes = [d for d in os.listdir("data/raw/plantvillage") if not d.startswith(".")]
print(f"✅ Successfully downloaded {len(classes)} classes!")
!find data/raw/plantvillage -type f | wc -l

# Recreate the original notebook's sorted class IDs and stratified 70/15/15 manifests.
# This is data preparation only; BASELINE_CKPT remains the already-trained model.
class_dirs = sorted([d for d in PV.iterdir() if d.is_dir() and not d.name.startswith('.')])
class_to_idx = {d.name:i for i,d in enumerate(class_dirs)}
idx_to_class = {i:d.name for i,d in enumerate(class_dirs)}
assert len(class_dirs) == 38, f'Expected 38 PlantVillage classes, found {len(class_dirs)}.'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
with open(SPLITS_DIR/'classes.json','w') as f:
    json.dump({'class_to_idx':class_to_idx,'idx_to_class':idx_to_class,'num_classes':38},f,indent=2)
fruit_crops={'Apple','Cherry_(including_sour)','Grape','Orange','Peach','Pepper,_bell','Strawberry','Tomato'}
records=[]
for folder in class_dirs:
    cname=folder.name; crop=cname.split('___')[0]
    for img in folder.iterdir():
        if img.suffix.lower() in {'.jpg','.jpeg','.png'}:
            records.append({'image_path':str(img.resolve()),
                'relative_path':str(img.relative_to(DATA)), 'class_name':cname,
                'class_idx':class_to_idx[cname], 'crop':crop,
                'category':'Fruit' if crop in fruit_crops else 'Plant',
                'is_healthy':'healthy' in cname.lower()})
frame=pd.DataFrame(records)
train_df,tmp_df=train_test_split(frame,test_size=.30,stratify=frame.class_idx,random_state=42)
val_df,test_df=train_test_split(tmp_df,test_size=.50,stratify=tmp_df.class_idx,random_state=42)
train_df.to_csv(SPLITS_DIR/'train.csv',index=False)
val_df.to_csv(SPLITS_DIR/'val.csv',index=False)
test_df.to_csv(SPLITS_DIR/'test.csv',index=False)
print(f'✅ Splits: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
assert (SPLITS_DIR/'classes.json').is_file()
assert all((SPLITS_DIR/f'{s}.csv').is_file() for s in ('train','val','test'))


In [ ]:
# Step 3: Download PlantDoc, PlantWild v1, and PlantSeg
# PlantVillage was downloaded by the original Step 2 cell above.
subprocess.run([sys.executable,"-m","pip","-q","install","huggingface_hub","imagehash","scikit-learn","seaborn"],check=True)
PLANTDOC = RAW / 'plantdoc'
if not PLANTDOC.exists():
    import urllib.request
    url='https://github.com/pratikkayal/PlantDoc-Dataset/archive/5467f6012d78d1c446145d5f582da6096f852ae8.zip'
    z=RAW/'plantdoc.zip'; urllib.request.urlretrieve(url,z)
    with zipfile.ZipFile(z) as f: f.extractall(RAW/'plantdoc_extract')
    roots=list((RAW/'plantdoc_extract').glob('PlantDoc-Dataset-*'))
    assert roots, 'PlantDoc archive structure was not recognized'
    shutil.move(str(roots[0]),str(PLANTDOC))
PLANTWILD=RAW/'plantwild'
if not PLANTWILD.exists():
    from huggingface_hub import hf_hub_download
    z=Path(hf_hub_download(repo_id='uqtwei2/PlantWild',repo_type='dataset',filename='plantwild.zip',local_dir=str(RAW/'hf_cache')))
    PLANTWILD.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(z) as f: f.extractall(PLANTWILD)
PLANTSEG=RAW/'plantseg'
if not PLANTSEG.exists():
    import urllib.request
    PLANTSEG.mkdir(parents=True,exist_ok=True)
    z=RAW/'plantsegv2.zip'
    urllib.request.urlretrieve('https://zenodo.org/records/13958858/files/plantsegv2.zip?download=1',z)
    with zipfile.ZipFile(z) as f: f.extractall(PLANTSEG)
print('PlantDoc:',PLANTDOC,'| PlantWild:',PLANTWILD,'| PlantSeg:',PLANTSEG)


In [ ]:
# Step 4: Inspect compatible labels and map them to the existing 38 classes
import unicodedata
from collections import Counter
with open(SPLITS_DIR/'classes.json') as f: class_meta=json.load(f)
class_to_idx={str(k):int(v) for k,v in class_meta['class_to_idx'].items()}
idx_to_class={int(k):str(v) for k,v in class_meta['idx_to_class'].items()}
assert len(class_to_idx)==38 and all(idx_to_class[v]==k for k,v in class_to_idx.items())

def norm(s):
    s=unicodedata.normalize('NFKD',str(s)).encode('ascii','ignore').decode().lower()
    return re.sub(r'[^a-z0-9]+','',s)

def map_label(label):
    raw=str(label).replace('___',' ').replace('_',' ').replace('+',' ')
    t=norm(raw)
    aliases={
      'applescableaf':'Apple___Apple_scab','appleleaf':'Apple___healthy','applerustleaf':'Apple___Cedar_apple_rust',
      'bellpepperleaf':'Pepper,_bell___healthy','bellpepperleafspot':'Pepper,_bell___Bacterial_spot',
      'bellpepperbacterialspotleaf':'Pepper,_bell___Bacterial_spot','bellpepperhealthy':'Pepper,_bell___healthy',
      'blueberryleaf':'Blueberry___healthy','cherryleaf':'Cherry_(including_sour)___healthy',
      'cornmaizegrayleafspot':'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
      'corngrayleafspot':'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
      'cornrustleaf':'Corn_(maize)___Common_rust_','peachleaf':'Peach___healthy',
      'potatoleafearlyblight':'Potato___Early_blight','potatoleaflateblight':'Potato___Late_blight',
      'raspberryleaf':'Raspberry___healthy','soyabeanleaf':'Soybean___healthy','soybeanleaf':'Soybean___healthy',
      'squashpowderymildewleaf':'Squash___Powdery_mildew','strawberryleaf':'Strawberry___healthy',
      'tomatoleaf':'Tomato___healthy','tomatoleafbacterialspot':'Tomato___Bacterial_spot',
      'tomatoleafearlyblight':'Tomato___Early_blight','tomatoleaflateblight':'Tomato___Late_blight',
      'tomatoleafmold':'Tomato___Leaf_Mold','tomatoseptorialeafspot':'Tomato___Septoria_leaf_spot',
      'tomatotargetspot':'Tomato___Target_Spot',
      'tomatotwospottedspidermitesleaf':'Tomato___Spider_mites Two-spotted_spider_mite',
      'tomatoyellowleafcurlvirus':'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
      'tomatomosaicvirus':'Tomato___Tomato_mosaic_virus'}
    if t in aliases and aliases[t] in class_to_idx: return aliases[t]
    matches=[c for c in class_to_idx if norm(c)==t]
    if len(matches)==1: return matches[0]
    t=t.replace('corn','cornmaize').replace('bellpepper','pepperbell').replace('soyabean','soybean')
    matches=[c for c in class_to_idx if norm(c)==t]
    return matches[0] if len(matches)==1 else None

def class_dirs(root):
    return sorted({p.parent.name for p in root.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}})
for dataset_root in (PLANTDOC,PLANTWILD):
    labels=class_dirs(dataset_root)
    pairs=[(x,map_label(x)) for x in labels]
    print(f'\n{dataset_root.name}: mapped {sum(v is not None for _,v in pairs)}/{len(pairs)} folder labels')
    print(pd.DataFrame(pairs,columns=['source_label','project_label']).to_string(index=False))


In [ ]:
# 3. Index images; deduplicate/group before splitting; never use external test images for training
from PIL import Image, ImageOps
import imagehash
from sklearn.model_selection import GroupShuffleSplit, train_test_split
IMG_EXT={'.jpg','.jpeg','.png','.bmp','.webp'}

def index_class_folders(root, source):
    rows=[]
    for p in root.rglob('*'):
        if p.suffix.lower() not in IMG_EXT: continue
        # Most dataset layouts use class/image. Keep nearest parent folder as label.
        label=p.parent.name; target=map_label(label)
        if target is None: continue
        try:
            with Image.open(p) as im:
                im.verify()
            with Image.open(p) as im:
                ph=str(imagehash.phash(ImageOps.exif_transpose(im).convert('RGB')))
        except Exception: continue
        rows.append({'image_path':str(p),'source':source,'source_label':label,
                     'class_name':target,'class_idx':class_to_idx[target],
                     'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'phash':ph,
                     'source_group':f'{source}:{norm(p.stem)}'})
    return rows

pv_rows=[]
for p in PV.rglob('*'):
    if p.suffix.lower() in IMG_EXT and p.parent.name in class_to_idx:
        target=p.parent.name
        pv_rows.append({'image_path':str(p),'source':'plantvillage','source_label':target,
          'class_name':target,'class_idx':class_to_idx[target],
          'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),
          'phash':str(imagehash.phash(Image.open(p).convert('RGB'))),
          'source_group':f'pv:{norm(p.stem)}'})
external=index_class_folders(PLANTDOC,'plantdoc')+index_class_folders(PLANTWILD,'plantwild')
# Use original image/source URLs from sidecar tables when present; retain basename grouping otherwise.
source_lookup={}
for root in (PLANTDOC,PLANTWILD):
    for meta_path in list(root.rglob('*.csv'))+list(root.rglob('*.tsv')):
        try:
            tab=pd.read_csv(meta_path,sep='\t' if meta_path.suffix=='.tsv' else ',')
            cols={norm(c):c for c in tab.columns}
            url_col=next((cols[k] for k in ('sourceurl','imageurl','originalurl','url') if k in cols),None)
            file_col=next((cols[k] for k in ('image','imagepath','filename','filepath','file') if k in cols),None)
            if url_col and file_col:
                for _,row in tab[[file_col,url_col]].dropna().iterrows():
                    source_lookup[norm(Path(str(row[file_col])).name)]=str(row[url_col])
        except Exception: pass
for row in external:
    url=source_lookup.get(norm(Path(row['image_path']).name))
    if url: row['source_group']='url:'+norm(url)
pv=pd.DataFrame(pv_rows); ext=pd.DataFrame(external)
assert len(pv)>0 and len(ext)>0, 'No compatible image records: inspect class mapping/layout above.'
print('PV rows:',len(pv),'compatible external rows:',len(ext))
print('External mapped counts by source/class:',ext.groupby(['source','class_name']).size().to_string())

# Group exact duplicates globally; group same basename within a source; perceptual hash candidate buckets
# catch common resized/re-encoded near-duplicates. Any identified group stays in one partition.
all_df=pd.concat([pv,ext],ignore_index=True).drop_duplicates('sha256').reset_index(drop=True)
parent=list(range(len(all_df)))
def find(a):
    while parent[a]!=a:
        parent[a]=parent[parent[a]]; a=parent[a]
    return a
def union(a,b):
    ra,rb=find(a),find(b)
    if ra!=rb: parent[rb]=ra
for _,ids in all_df.groupby('source_group').groups.items():
    ids=list(ids)
    for j in ids[1:]: union(ids[0],j)
# Four 16-bit bands. Hamming-distance <=3 must collide in at least one band.
buckets={}
for i,h in enumerate(all_df.phash.astype(str)):
    h=h.zfill(16)
    for b in range(4):
        key=(b,h[b*4:(b+1)*4])
        for j in buckets.get(key,[]):
            if sum(x!=y for x,y in zip(h,all_df.at[j,'phash'].zfill(16)))<=6: union(i,j)
        buckets.setdefault(key,[]).append(i)
all_df['dup_group']=[find(i) for i in range(len(all_df))]
# Load the project's exact original split manifests. This preserves the same PV test set used for the 93.15% result.
def load_original_split(name):
    old=pd.read_csv(ORIGINAL_SPLITS/f'{name}.csv')
    mapped_paths=[]
    for _,r in old.iterrows():
        rel=Path(str(r.get('relative_path','')))
        # Existing split relative_path is data/raw/plantvillage/<class>/<file>.
        if len(rel.parts)>=4 and rel.parts[0]=='raw' and rel.parts[1]=='plantvillage':
            path=PV.joinpath(*rel.parts[2:])
        else:
            path=Path(str(r['image_path']))
            if not path.is_file(): path=PV/r['class_name']/Path(str(r['image_path'])).name
        mapped_paths.append(str(path))
    old['image_path']=mapped_paths
    old['class_idx']=old['class_name'].map(class_to_idx)
    old=old[old.class_idx.notna() & old.image_path.map(lambda x:Path(x).is_file())].copy()
    old['class_idx']=old.class_idx.astype(int); old['source']='plantvillage'
    old['sha256']=old.image_path.map(lambda x:hashlib.sha256(Path(x).read_bytes()).hexdigest())
    old['phash']=old.image_path.map(lambda x:str(imagehash.phash(Image.open(x).convert('RGB'))))
    old['source_group']=old.image_path.map(lambda x:'pv:'+norm(Path(x).stem))
    return old
pv_train=load_original_split('train'); pv_val=load_original_split('val'); pv_test=load_original_split('test')
assert len(pv_test)==8146 and pv_test.class_idx.nunique()==38, \
    f'Original PlantVillage test split incomplete: {len(pv_test)} images / {pv_test.class_idx.nunique()} classes; expected 8,146 / 38.'
# Assign each PV test image its global duplicate group and remove same/near-duplicates from adaptation.
pv_group_by_sha=all_df.drop_duplicates('sha256').set_index('sha256').dup_group.to_dict()
pv_train['dup_group']=pv_train.sha256.map(pv_group_by_sha); pv_val['dup_group']=pv_val.sha256.map(pv_group_by_sha); pv_test['dup_group']=pv_test.sha256.map(pv_group_by_sha)
pv_test_groups=set(pv_test.dup_group)
pv_val_groups=set(pv_val.dup_group)
pv_all_groups=set(pv_train.dup_group)|pv_val_groups|pv_test_groups
# Remove any external image duplicated against any PV partition, keeping datasets domain-independent.
ext_part=all_df[(all_df.source!='plantvillage') & ~all_df.dup_group.isin(pv_all_groups)].copy()
ext_groups=ext_part.drop_duplicates('dup_group')
train_g=[]; val_g=[]; test_g=[]
for cls,g in ext_groups.groupby('class_idx'):
    gs=g.dup_group.to_numpy()
    if len(gs)<3: train_g.extend(gs); continue
    g_train,g_test=train_test_split(gs,test_size=max(1,round(.20*len(gs))),random_state=SEED)
    if len(g_train)>=2: g_train,g_val=train_test_split(g_train,test_size=max(1,round(.15*len(g_train))),random_state=SEED)
    else: g_val=[]
    train_g.extend(g_train); val_g.extend(g_val); test_g.extend(g_test)
field_train=ext_part[ext_part.dup_group.isin(train_g)].copy()
field_val=ext_part[ext_part.dup_group.isin(val_g)].copy()
field_test_df=ext_part[ext_part.dup_group.isin(test_g)].copy()
# If any original PV test has a close duplicate inside the PV training/validation manifests, remove that copy.
pv_train=pv_train[~pv_train.dup_group.isin(pv_val_groups|pv_test_groups)].copy()
pv_val=pv_val[~pv_val.dup_group.isin(pv_test_groups)].copy()
train_df=pd.concat([pv_train,field_train],ignore_index=True)
val_df=pd.concat([pv_val,field_val],ignore_index=True)
assert set(train_df.dup_group if 'dup_group' in train_df else []).isdisjoint(set(field_test_df.dup_group)), 'Duplicate leakage detected'
assert not set(pv_train.sha256).intersection(pv_test.sha256), 'Exact PV duplicate leakage detected'
assert not set(field_test_df.sha256).intersection(pv_test.sha256), 'Field/PV test duplicate leakage detected'
train_df.to_csv(OUT/'adapt_train.csv',index=False); val_df.to_csv(OUT/'adapt_val.csv',index=False)
pv_test.to_csv(OUT/'plantvillage_test.csv',index=False); field_test_df.to_csv(OUT/'field_test.csv',index=False)
print('Train / validation:',len(train_df),len(val_df),'| PV held-out:',len(pv_test),'| field held-out:',len(field_test_df))
print('Field test source counts:',field_test_df.source.value_counts().to_dict())


In [ ]:
# 4. Data loaders with aligned augmentation, class balancing, and validation/test left untouched
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
MEAN=[.485,.456,.406]; STD=[.229,.224,.225]; SIZE=224
train_tf=transforms.Compose([transforms.Resize((SIZE,SIZE)),transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=.2),transforms.RandomRotation(20),
    transforms.ColorJitter(.25,.25,.25,.08),transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),transforms.RandomErasing(p=.20,scale=(.02,.12))])
eval_tf=transforms.Compose([transforms.Resize((SIZE,SIZE)),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
class ImageTable(Dataset):
    def __init__(self,df,tf): self.df=df.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        with Image.open(r.image_path) as im: x=self.tf(ImageOps.exif_transpose(im).convert('RGB'))
        return x,int(r.class_idx)
counts=train_df.class_idx.value_counts().to_dict()
sample_weights=torch.DoubleTensor([1.0/counts[int(c)] for c in train_df.class_idx])
sampler=WeightedRandomSampler(sample_weights,len(sample_weights),replacement=True)
BATCH=32
train_loader=DataLoader(ImageTable(train_df,train_tf),batch_size=BATCH,sampler=sampler,num_workers=2,pin_memory=True)
val_loader=DataLoader(ImageTable(val_df,eval_tf),batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
pv_test_loader=DataLoader(ImageTable(pv_test,eval_tf),batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
field_test_loader=DataLoader(ImageTable(field_test_df,eval_tf),batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
print('Class-balanced sampler active; augmentation is train-only.')


In [ ]:
# 5. Restore the existing EfficientNet-B0 classifier checkpoint; attach lesion-mask decoder
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b0
base=efficientnet_b0(weights=None)
base.classifier=nn.Sequential(nn.Dropout(.2),nn.Linear(base.classifier[1].in_features,38))
ckpt=torch.load(BASELINE_BACKUP,map_location='cpu',weights_only=False)
state=ckpt.get('model_state_dict',ckpt.get('state_dict',ckpt))
# Accept both this repo's CropDiseaseClassifier keys and torchvision-style notebook keys.
if any(k.startswith('backbone.') for k in state): state={k.removeprefix('backbone.'):v for k,v in state.items() if k.startswith('backbone.')}
msg=base.load_state_dict(state,strict=True)
assert base.classifier[-1].out_features==len(class_to_idx)

class EfficientNetB0MultiTask(nn.Module):
    def __init__(self, backbone):
        super().__init__(); self.backbone=backbone
        self.seg=nn.Sequential(nn.Conv2d(1280,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU(inplace=True),
          nn.Conv2d(256,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(inplace=True),nn.Conv2d(128,1,1))
    def forward(self,x):
        feat=self.backbone.features(x)
        pooled=self.backbone.avgpool(feat).flatten(1)
        logits=self.backbone.classifier(pooled)
        mask_logits=F.interpolate(self.seg(feat),size=x.shape[-2:],mode='bilinear',align_corners=False)
        return logits,mask_logits
model=EfficientNetB0MultiTask(base).to(DEVICE)
print('Loaded original 38-way classifier. New lesion decoder initialized separately.')


In [ ]:
# Step 6 (optional): Pair PlantSeg images with PNG masks and train the segmentation decoder
TRAIN_SEGMENTATION = False  # Leave False to skip this stage and continue classification fine-tuning.
# The archive stores masks under Annotations/ and split folders, with images under Images/;
# the JSON files are COCO/LabelMe metadata and are not one JSON file beside every image.
from PIL import Image

def split_name(path,root):
    parts=[x.lower() for x in Path(path).relative_to(root).parts[:-1]]
    for part in parts:
        if part in {'train','training'}: return 'train'
        if part in {'val','valid','validation'}: return 'val'
        if part in {'test','testing'}: return 'test'
    return None

def mask_key(path):
    stem=Path(path).stem.lower()
    for suffix in ('_mask','_label','_annotation','_ann'):
        if stem.endswith(suffix): stem=stem[:-len(suffix)]
    return stem

image_files=[p for p in PLANTSEG.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT
             and any(part.lower() in {'images','image'} for part in p.parts)]
mask_files=[p for p in PLANTSEG.rglob('*.png') if p.is_file()
            and any(part.lower() in {'annotations','annotation','masks','mask'} for part in p.parts)]
mask_by_key={}
for mp in mask_files:
    key=(split_name(mp,PLANTSEG),mask_key(mp))
    mask_by_key.setdefault(key,mp)
seg_pairs=[]
for ip in image_files:
    split=split_name(ip,PLANTSEG); key=mask_key(ip)
    mp=mask_by_key.get((split,key)) or mask_by_key.get((None,key))
    if mp is not None: seg_pairs.append({'image':str(ip),'mask':str(mp),'split':split})
print('PlantSeg directories:', sorted({str(p.parent) for p in image_files[:3]}), '| masks found:',len(mask_files))
print('PlantSeg image/mask pairs:',len(seg_pairs),'| official splits:',pd.Series([r['split'] for r in seg_pairs]).value_counts(dropna=False).to_dict())
if TRAIN_SEGMENTATION:
    assert len(seg_pairs)>0, 'No PlantSeg image/PNG mask pairs found. Inspect the extracted archive folders.'
    seg_meta=pd.DataFrame(seg_pairs)
    # Preserve PlantSeg's released train/validation/test folders when present.
    if {'train','val'}.issubset(set(seg_meta.split.dropna().unique())):
        seg_train=seg_meta[seg_meta.split=='train'].copy()
        seg_val=seg_meta[seg_meta.split=='val'].copy()
    else:
        # Fallback for archive variants without named split folders.
        ph=[]
        for ip in seg_meta.image:
            with Image.open(ip) as im: ph.append(str(imagehash.phash(ImageOps.exif_transpose(im).convert('RGB'))))
        seg_meta['duplicate_group']=ph
        groups=seg_meta.drop_duplicates('duplicate_group')
        tr_groups,va_groups=train_test_split(groups.duplicate_group.to_numpy(),test_size=.2,random_state=SEED)
        seg_train=seg_meta[seg_meta.duplicate_group.isin(tr_groups)].copy()
        seg_val=seg_meta[seg_meta.duplicate_group.isin(va_groups)].copy()
    assert len(seg_train)>0 and len(seg_val)>0, 'PlantSeg train/validation masks are empty; inspect split folder names.'

    class SegTable(Dataset):
        def __init__(self,df): self.df=df.reset_index(drop=True)
        def __len__(self): return len(self.df)
        def __getitem__(self,i):
            r=self.df.iloc[i]
            with Image.open(r['image']) as im: im=ImageOps.exif_transpose(im).convert('RGB').resize((SIZE,SIZE))
            with Image.open(r['mask']) as m:
                m=ImageOps.exif_transpose(m).convert('L').resize((SIZE,SIZE),Image.Resampling.NEAREST)
                mask=(np.asarray(m)>0).astype(np.float32)
            x=transforms.ToTensor()(im); x=transforms.Normalize(MEAN,STD)(x)
            y=torch.from_numpy(mask).unsqueeze(0)
            return x,y
    seg_train_loader=DataLoader(SegTable(seg_train),batch_size=16,shuffle=True,num_workers=2,pin_memory=True)
    seg_val_loader=DataLoader(SegTable(seg_val),batch_size=16,shuffle=False,num_workers=2,pin_memory=True)

    def dice_loss(logits,target,eps=1e-6):
        p=logits.sigmoid(); inter=(p*target).sum((2,3)); denom=p.sum((2,3))+target.sum((2,3))
        return 1-((2*inter+eps)/(denom+eps)).mean()
    # Freeze classifier; optimize decoder + late feature blocks using diseased-region masks.
    for p in model.backbone.classifier.parameters(): p.requires_grad=False
    for i,b in enumerate(model.backbone.features):
        for p in b.parameters(): p.requires_grad=(i>=5)
    seg_opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=2e-4,weight_decay=1e-4)
    for epoch in range(1,6):
        model.train(); losses=[]
        for x,y in seg_train_loader:
            x,y=x.to(DEVICE),y.to(DEVICE); seg_opt.zero_grad()
            _,m=model(x); loss=F.binary_cross_entropy_with_logits(m,y)+dice_loss(m,y)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); seg_opt.step(); losses.append(loss.item())
        model.eval(); vals=[]
        with torch.no_grad():
            for x,y in seg_val_loader:
                x,y=x.to(DEVICE),y.to(DEVICE); _,m=model(x); vals.append((F.binary_cross_entropy_with_logits(m,y)+dice_loss(m,y)).item())
        print(f'Seg epoch {epoch}: train loss={np.mean(losses):.4f}, val loss={np.mean(vals):.4f}')
    # PlantSeg masks cover diseased tissue, not the full leaf silhouette.

else:
    print('Skipping PlantSeg segmentation training. Classification fine-tuning will continue; segmentation decoder remains untrained.')


In [ ]:
# 7. Fine-tune classifier using PV train + mapped field-domain train; early stop and lower LR on plateau
from sklearn.metrics import f1_score
for p in model.backbone.classifier.parameters(): p.requires_grad=True
# Warm start all feature blocks frozen, then progressively unfreeze upper blocks.
for p in model.backbone.features.parameters(): p.requires_grad=False
criterion=nn.CrossEntropyLoss(label_smoothing=0.05)
opt=torch.optim.AdamW(model.backbone.classifier.parameters(),lr=2e-4,weight_decay=5e-4)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max',factor=.5,patience=1,min_lr=1e-6)
scaler=torch.amp.GradScaler('cuda')

def infer(loader):
    model.eval(); ls=[]; ys=[]
    with torch.no_grad():
        for x,y in loader:
            x=x.to(DEVICE); logits,_=model(x); ls.append(logits.float().cpu()); ys.append(y)
    return torch.cat(ls).numpy(),torch.cat(ys).numpy()

def epoch_pass(loader,training):
    model.train(training); losses=[]; pred=[]; true=[]
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        if training:
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16): logits,_=model(x); loss=criterion(logits,y)
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update()
        else:
            with torch.no_grad(): logits,_=model(x); loss=criterion(logits,y)
        losses.append(loss.item()); pred.extend(logits.argmax(1).detach().cpu().tolist()); true.extend(y.cpu().tolist())
    return float(np.mean(losses)),float(np.mean(np.array(pred)==np.array(true))),f1_score(true,pred,average='macro',zero_division=0)

best_f1=-1; patience=3; stale=0; history=[]
for epoch in range(1,16):
    # unfreeze blocks 6-8 after first two classifier-only epochs
    if epoch==3:
        for i,b in enumerate(model.backbone.features):
            if i>=6:
                for p in b.parameters(): p.requires_grad=True
        opt=torch.optim.AdamW([{'params':[p for p in model.backbone.features.parameters() if p.requires_grad],'lr':2e-5},
                               {'params':model.backbone.classifier.parameters(),'lr':1e-4}],weight_decay=5e-4)
        scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max',factor=.5,patience=1,min_lr=1e-6)
    tr_loss,tr_acc,tr_f1=epoch_pass(train_loader,True)
    va_loss,va_acc,va_f1=epoch_pass(val_loader,False); scheduler.step(va_f1)
    history.append({'epoch':epoch,'train_loss':tr_loss,'train_accuracy':tr_acc,'train_macro_f1':tr_f1,
                    'val_loss':va_loss,'val_accuracy':va_acc,'val_macro_f1':va_f1,'lr':opt.param_groups[-1]['lr']})
    print(f'E{epoch:02d} train acc {tr_acc:.4f} loss {tr_loss:.4f} | val acc {va_acc:.4f} macroF1 {va_f1:.4f} loss {va_loss:.4f} | lr {opt.param_groups[-1]["lr"]:.2g}')
    if va_f1>best_f1:
        best_f1=va_f1; stale=0
        torch.save({'model_state_dict':model.state_dict(),'class_to_idx':class_to_idx,'idx_to_class':idx_to_class,
                    'val_macro_f1':best_f1,'epoch':epoch},OUT/'field_finetuned_best.pth')
    else:
        stale+=1
        if stale>=patience:
            print('Early stopping: validation macro-F1 stopped improving.')
            break
pd.DataFrame(history).to_csv(OUT/'training_history.csv',index=False)
# Compare train/validation curves; widening gap or rising val loss is the overfit signal.
import matplotlib.pyplot as plt
hist=pd.DataFrame(history)
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].plot(hist.epoch,hist.train_accuracy,label='train');ax[0].plot(hist.epoch,hist.val_accuracy,label='validation');ax[0].legend();ax[0].set_title('Accuracy')
ax[1].plot(hist.epoch,hist.train_loss,label='train');ax[1].plot(hist.epoch,hist.val_loss,label='validation');ax[1].legend();ax[1].set_title('Loss');plt.show()


In [ ]:
# 8. Calibrate confidence on validation only; full classification metrics on separate test domains
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,precision_recall_fscore_support,
    classification_report,confusion_matrix,roc_auc_score,average_precision_score,log_loss,matthews_corrcoef,cohen_kappa_score)
from scipy.optimize import minimize_scalar
from sklearn.preprocessing import label_binarize
# Restore best adapted weights; segmentation decoder remains in the same artifact.
best=torch.load(OUT/'field_finetuned_best.pth',map_location=DEVICE,weights_only=False)
model.load_state_dict(best['model_state_dict'])
val_logits,val_y=infer(val_loader)
def softmax_np(z,T=1.):
    a=z/T; a=a-a.max(axis=1,keepdims=True); e=np.exp(a); return e/e.sum(axis=1,keepdims=True)
# Temperature scaling is fitted only on validation data (no test leakage).
T=float(minimize_scalar(lambda t:log_loss(val_y,softmax_np(val_logits,t),labels=np.arange(38)),bounds=(.05,10),method='bounded').x)
print('Validation-fitted temperature:',T)

def evaluate_domain(name,loader,df,temperature=T):
    logits,y=infer(loader); probs=softmax_np(logits,temperature); pred=probs.argmax(1)
    p,r,f1,s=precision_recall_fscore_support(y,pred,labels=np.arange(38),zero_division=0)
    present=np.unique(y); yb=label_binarize(y,classes=np.arange(38))
    auc=np.nan; pr_auc=np.nan
    if len(present)>1:
        auc=roc_auc_score(yb[:,present],probs[:,present],average='macro',multi_class='ovr')
        pr_auc=average_precision_score(yb[:,present],probs[:,present],average='macro')
    # ECE uses 15 equal-width confidence bins; confidence is max calibrated class probability.
    conf=probs.max(1); correct=(pred==y); ece=0.
    for lo in np.linspace(0,1,16)[:-1]:
        m=(conf>=lo)&(conf<(lo+1/15))
        if m.any(): ece+=m.mean()*abs(correct[m].mean()-conf[m].mean())
    # healthy-vs-diseased derived from the class names, not a second learned head.
    healthy=np.array(['healthy' in idx_to_class[int(i)].lower() for i in y])
    pred_healthy=np.array(['healthy' in idx_to_class[int(i)].lower() for i in pred])
    bprec,brec,bf1,_=precision_recall_fscore_support(healthy,pred_healthy,average='binary',zero_division=0)
    cm=confusion_matrix(y,pred,labels=np.arange(38)); specificity=[]
    for i in range(38):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=cm.sum()-tp-fn-fp
        specificity.append(tn/max(1,tn+fp))
    metrics={'domain':name,'n':int(len(y)),'accuracy':accuracy_score(y,pred),'test_accuracy':accuracy_score(y,pred),'balanced_accuracy':balanced_accuracy_score(y,pred),
      'precision_macro':float(np.mean(p[present])),'recall_macro':float(np.mean(r[present])),'f1_macro':float(np.mean(f1[present])),
      'f1_weighted':float(f1_score(y,pred,average='weighted',zero_division=0)),'f1_micro':float(f1_score(y,pred,average='micro')),
      'mcc':matthews_corrcoef(y,pred),'cohen_kappa':cohen_kappa_score(y,pred),'specificity_macro':float(np.mean(np.array(specificity)[present])),
      'roc_auc_ovr_macro':float(auc),'pr_auc_macro':float(pr_auc),'log_loss':float(log_loss(y,probs,labels=np.arange(38))),
      'brier_multiclass':float(np.mean(np.sum((probs-np.eye(38)[y])**2,axis=1))),
      'mean_confidence':float(conf.mean()),'confidence_ece_15bins':float(ece),'temperature':temperature,
      'healthy_infected_f1':float(bf1),'healthy_infected_precision':float(bprec),'healthy_infected_recall':float(brec),
      'per_class':classification_report(y,pred,labels=np.arange(38),target_names=[idx_to_class[i] for i in range(38)],output_dict=True,zero_division=0)}
    (OUT/f'{name.lower().replace(" ","_")}_metrics.json').write_text(json.dumps(metrics,indent=2))
    pd.DataFrame(cm,index=[idx_to_class[i] for i in range(38)],columns=[idx_to_class[i] for i in range(38)]).to_csv(OUT/f'{name.lower().replace(" ","_")}_confusion_matrix.csv')
    print(name, {k:round(v,4) for k,v in metrics.items() if isinstance(v,float)})
    return metrics

# PlantVillage benchmark: untouched 15% test split. Field-domain: untouched grouped PlantDoc/PlantWild test.
# Baseline and adapted are both scored on each same test set for a fair delta.
def score_classifier(state,loader):
    model.load_state_dict(state,strict=True); return infer(loader)
# adapted model reports
pv_adapt=evaluate_domain('PlantVillage adapted',pv_test_loader,pv_test)
field_adapt=evaluate_domain('Field-like adapted',field_test_loader,field_test_df)
# Evaluate baseline by loading checkpoint into EfficientNet base and forwarding; decoder not needed.
model.load_state_dict(best['model_state_dict'])
base_state=state
# store adapted logits before swapping; then evaluate original base with compatible state dict
adapt_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
adapt_train_eval_loader=DataLoader(ImageTable(train_df,eval_tf),batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
adapt_train_logits,adapt_train_y=infer(adapt_train_eval_loader)
adapt_train_acc=float((adapt_train_logits.argmax(1)==adapt_train_y).mean())
adapt_val_acc=float((val_logits.argmax(1)==val_y).mean())
base_state={k:v.detach().cpu().clone() for k,v in model.backbone.state_dict().items()}
orig=torch.load(BASELINE_BACKUP,map_location='cpu',weights_only=False); orig_state=orig.get('model_state_dict',orig.get('state_dict',orig))
if any(k.startswith('backbone.') for k in orig_state): orig_state={k.removeprefix('backbone.'):v for k,v in orig_state.items() if k.startswith('backbone.')}
base_model=model.backbone
base_model.load_state_dict(orig_state,strict=True); base_model.eval()
# Baseline confidence temperature is also fitted on validation only.
class BaseOnly(nn.Module):
    def __init__(self,b): super().__init__(); self.b=b
    def forward(self,x): return self.b(x), torch.empty(0,device=x.device)
model=BaseOnly(base_model).to(DEVICE)
base_val_logits,base_val_y=infer(val_loader)
base_T=float(minimize_scalar(lambda t:log_loss(base_val_y,softmax_np(base_val_logits,t),labels=np.arange(38)),bounds=(.05,10),method='bounded').x)
# Consistent accuracy reporting for the original and adapted model.
train_eval_loader=DataLoader(ImageTable(train_df,eval_tf),batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
base_train_logits,base_train_y=infer(train_eval_loader); base_train_acc=float((base_train_logits.argmax(1)==base_train_y).mean())
base_val_acc=float((base_val_logits.argmax(1)==base_val_y).mean())
# classification-only forward adapter for same inference helper
class BaseWrapper(nn.Module):
    def __init__(self,b): super().__init__(); self.b=b
    def forward(self,x): return self.b(x), torch.empty(0,device=x.device)
# BaseOnly already wraps the original classifier and returns logits in infer()'s expected format.
# Avoid creating a fresh 1,000-class EfficientNet here; that unused model cannot load 38-class weights.
base_pv=evaluate_domain('PlantVillage baseline',pv_test_loader,pv_test,temperature=base_T)
base_field=evaluate_domain('Field-like baseline',field_test_loader,field_test_df,temperature=base_T)
for m in (base_pv,base_field):
    m['train_accuracy']=base_train_acc; m['validation_accuracy']=base_val_acc
for m in (pv_adapt,field_adapt):
    m['train_accuracy']=adapt_train_acc; m['validation_accuracy']=adapt_val_acc
for m in (base_pv,pv_adapt,base_field,field_adapt):
    (OUT/f"{m['domain'].lower().replace(' ','_')}_metrics.json").write_text(json.dumps(m,indent=2))
# Persist a clear comparison table.
rows=[]
for m in (base_pv,pv_adapt,base_field,field_adapt):
    rows.append({k:v for k,v in m.items() if k!='per_class'})
pd.DataFrame(rows).to_csv(OUT/'baseline_vs_adapted_summary.csv',index=False)
print('Artifacts:',OUT)


## Reading the evaluation

- `PlantVillage baseline` vs `PlantVillage adapted` measures whether domain fine-tuning damaged the original benchmark.
- `Field-like baseline` vs `Field-like adapted` is the key real-world-domain comparison. These are held-out PlantDoc/PlantWild groups, not training images.
- Metrics include train/validation/test accuracy, macro and weighted precision/recall/F1, balanced accuracy, specificity, MCC, Cohen kappa, one-vs-rest ROC AUC, macro average precision (PR AUC), log loss, multiclass Brier score, per-class report, and confusion matrices. AUC is calculated only for labels represented in that test set.
- The adapted-model temperature is fitted on validation logits only. `mean_confidence` is the mean maximum calibrated softmax probability; `confidence_ece_15bins` measures confidence calibration error. Treat this as a calibrated score, not certainty.
- Class imbalance is handled with a weighted sampler plus train-only geometric/color augmentation and label smoothing. This avoids fabricating synthetic disease textures. Early stopping and ReduceLROnPlateau respond to validation macro-F1; compare train vs validation curves to detect overfitting.
- Duplicate control uses exact SHA-256, source/basename grouping, and perceptual-hash near-duplicate candidates before external splits. It cannot guarantee identifying all shared original sources; preserve/check source URLs and manually audit flagged pairs.
- PlantDoc and PlantWild labels that do not map unambiguously are excluded. Review the printed mapping table and per-class counts before trusting results. PlantWild is licensed non-commercially; the segmentation dataset and image rights also require checking for your use case.

In [ ]:
# 9. Export a full research artifact and a backend-compatible classifier checkpoint
FINAL=OUT/'efficientnet_b0_field_adapted_with_lesion_decoder.pth'
torch.save({'model_state_dict':adapt_state,'class_to_idx':class_to_idx,'idx_to_class':idx_to_class,
            'temperature':T,'segmentation_trained':bool(globals().get('TRAIN_SEGMENTATION',False)),
            'segmentation_target':'PlantSeg diseased-region masks',
            'baseline_checkpoint':str(BASELINE_CKPT),'seed':SEED},FINAL)

# The backend CropDiseaseClassifier expects only keys prefixed with backbone.*.
# Drop the optional decoder weights so this checkpoint loads strictly in backend/app/main.py.
DEPLOYMENT_CKPT=OUT/'best_model_field_adapted.pth'
deployment_state={k:v.detach().cpu() for k,v in adapt_state.items() if k.startswith('backbone.')}
assert deployment_state and all(k.startswith('backbone.') for k in deployment_state)
torch.save({'model_state_dict':deployment_state,'class_to_idx':class_to_idx,'idx_to_class':idx_to_class,
            'phase':'field_finetune','epoch':best.get('epoch'),
            'val_macro_f1':best.get('val_macro_f1'),'baseline_checkpoint':str(BASELINE_CKPT)},DEPLOYMENT_CKPT)
CLASSES_EXPORT=OUT/'classes.json'
shutil.copy2(SPLITS_DIR/'classes.json',CLASSES_EXPORT)
print('Full adapted artifact:',FINAL)
print('Backend-compatible classifier:',DEPLOYMENT_CKPT)
print('Class map:',CLASSES_EXPORT)
print('Baseline preserved at:',BASELINE_BACKUP)

# Download the adapted weights, matching class IDs, and baseline/adapted score comparison.
from google.colab import files
files.download(str(DEPLOYMENT_CKPT))
files.download(str(CLASSES_EXPORT))
summary_path=OUT/'baseline_vs_adapted_summary.csv'
if summary_path.is_file(): files.download(str(summary_path))


## Inspect held-out field-like examples

Run this after evaluation to inspect images from the untouched PlantDoc/PlantWild test split. The gallery reports each image’s true class, predicted class, and calibrated confidence, and creates a ZIP with the sampled images plus a CSV. These examples are for qualitative checking; do not use the held-out samples for further training or tuning.

In [ ]:
# 10. Field-like held-out image gallery and downloadable sample pack
from pathlib import Path
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import pandas as pd, numpy as np, torch, zipfile, shutil
from torchvision.models import efficientnet_b0
from torch import nn

# Reload the adapted checkpoint explicitly so this gallery is independent of whichever
# model (baseline/adapted) was last used by the evaluation cell.
artifact=torch.load(FINAL,map_location=DEVICE,weights_only=False)
gallery_backbone=efficientnet_b0(weights=None)
gallery_backbone.classifier=nn.Sequential(nn.Dropout(p=0.2),nn.Linear(1280,len(idx_to_class)))
gallery_model=EfficientNetB0MultiTask(gallery_backbone).to(DEVICE)
gallery_model.load_state_dict(artifact['model_state_dict'],strict=True)
gallery_model.eval()
gallery_temperature=float(artifact.get('temperature',1.0))

# One example per duplicate group, then sample up to four each from PlantDoc and PlantWild.
gallery_rows=[]
for source_name in ('plantdoc','plantwild'):
    pool=field_test_df[field_test_df.source==source_name].copy()
    if 'dup_group' in pool.columns: pool=pool.drop_duplicates('dup_group')
    if len(pool):
        k=min(4,len(pool))
        gallery_rows.append(pool.sample(n=k,random_state=SEED))
if not gallery_rows:
    raise ValueError('No PlantDoc/PlantWild held-out examples found in field_test_df.')
gallery_df=pd.concat(gallery_rows,ignore_index=True)
records=[]
for _,row in gallery_df.iterrows():
    with Image.open(row['image_path']) as im:
        pil=ImageOps.exif_transpose(im).convert('RGB')
        tensor=eval_tf(pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits,_=gallery_model(tensor)
        probs=torch.softmax(logits/gallery_temperature,dim=1)[0]
        conf,pred_idx=probs.max(dim=0)
    records.append({'source':row['source'],'image_path':row['image_path'],
        'true_class':row['class_name'],'predicted_class':idx_to_class[int(pred_idx)],
        'confidence':float(conf),'correct':idx_to_class[int(pred_idx)]==row['class_name'],
        'image':pil})

fig,axes=plt.subplots(len(records),1,figsize=(12,max(4,3.7*len(records))))
if len(records)==1: axes=[axes]
for ax,item in zip(axes,records):
    ax.imshow(item['image']); ax.axis('off')
    ok='✓' if item['correct'] else '✗'
    ax.set_title(f"{item['source']} | {ok} | true: {item['true_class']} | predicted: {item['predicted_class']} | confidence: {item['confidence']:.1%}",fontsize=10)
plt.tight_layout(); plt.show()

sample_dir=OUT/'field_test_gallery'; sample_dir.mkdir(parents=True,exist_ok=True)
zip_path=OUT/'heldout_field_test_samples.zip'
for i,item in enumerate(records,1):
    src=Path(item['image_path'])
    safe_true=str(item['true_class']).replace('/','_').replace('\\','_')
    dst=sample_dir/f"{i:02d}_{item['source']}_true-{safe_true}{src.suffix.lower()}"
    shutil.copy2(src,dst)
pred_df=pd.DataFrame([{k:v for k,v in item.items() if k!='image'} for item in records])
pred_csv=OUT/'heldout_field_test_sample_predictions.csv'
pred_df.to_csv(pred_csv,index=False)
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for image_file in sorted(sample_dir.iterdir()): zf.write(image_file,arcname=image_file.name)
print(pred_df.drop(columns=['image_path']).to_string(index=False))
print('Sample ZIP:',zip_path,'| prediction CSV:',pred_csv)
# Download the sampled held-out images and the matching prediction table from Colab.
from google.colab import files
files.download(str(zip_path))
files.download(str(pred_csv))